# Function 8: 8D black-box maximisation

You're optimising an eight-dimensional black-box function, where each of the eight input parameters affects the output, but the internal mechanics are unknown. Your objective is to find the parameter combination that maximises the function's output, such as performance, efficiency or validation accuracy. Because the function is high-dimensional and likely complex, global optimisation is hard, so identifying strong local maxima is often a practical strategy. For example, imagine you're tuning an ML model with eight hyperparameters: learning rate, batch size, number of layers, dropout rate, regularisation strength, activation function (numerically encoded), optimiser type (encoded) and initial weight range. Each input set returns a single validation accuracy score. Your goal is to maximise this score.

## How this notebook works

1. **Data loading:** the incumbent (best observed `y`, earliest round wins ties) and the latest observation are derived from `data.csv`.
2. **Tuned GP:** Matern 5/2 ARD Gaussian process with a Yeo-Johnson target transform; the noise floor (alpha) is tuned by grouped leave-one-out CV.
3. **Local trust-region search:** EI, PI, UCB and posterior mean under two kernels (Matern 5/2 and 3/2) in a box around the incumbent.
4. **Conservative refinement:** a tighter box around the cross-kernel consensus centre, penalising model disagreement, distance and uncertainty.
5. **Neural-network cross-check:** an MLP ensemble validated by leave-one-out CV, then used alongside the GPs in a final consensus search.
6. **Final recommendation:** a model-based candidate is submitted only if its predicted gain exceeds the tuned GP's leave-one-out MAE; otherwise the incumbent is submitted.

Everything is derived from `data.csv`: append new rows and re-run, with no code edits.

**Data-handling choices.**
- *Incumbent tie-break.* Observations with the same `y` (within `TIE_TOL`) are ties; the earliest round is the incumbent because that is where the optimum was first identified.
- *Replicates.* Exact repeat queries are kept. Cross-validation holds out all copies of an input together (grouped leave-one-out); otherwise a held-out point is predicted from its own twin and validation metrics are inflated.
- *Input format.* Portal inputs are rounded to 6 decimals and clipped to [0, 0.999999].

Attribution note: the neural-network cross-validation was drafted with ChatGPT's assistance.

In [1]:
# =============================================================================
# IMPORTS AND CONFIGURATION
# =============================================================================
# All tunable settings live in this cell. Every later cell reads from here, so
# model specifications and search settings stay consistent and reproducible
# (all random seeds are explicit).

import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, qmc
from sklearn.compose import TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, Matern
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, cross_val_predict
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# --- Data --------------------------------------------------------------------
DATA_PATH = "../data/data.csv"
FUNCTION_ID = 8
TIE_TOL = 1e-9               # y values within this tolerance count as tied for the incumbent
DECIMALS, UPPER_BOUND = 6, 0.999999   # portal input format: 6 decimals, inputs in [0, 0.999999]

# --- Search regions (unit-cube half-widths) ----------------------------------
WIDE_HALF_WIDTH = 0.10       # local trust region: incumbent +/- this
REFINE_HALF_WIDTH = 0.05     # refinement region: consensus centre +/- this

# --- Gaussian processes ------------------------------------------------------
GP_SEED = 42
TUNED_LS_BOUNDS = (1e-2, 1e4)        # ARD length-scale bounds (noise handled by alpha)
TUNED_RESTARTS = 25                  # optimiser restarts per GP fit
ALPHA_GRID = [1e-4, 1e-3, 1e-2, 1e-1]   # noise floors tried by grouped leave-one-out CV

# --- Acquisition and candidate search ---------------------------------------
SOBOL_POWER = 17                         # 2**17 candidates per search
SOBOL_SEED_WIDE, SOBOL_SEED_REFINE, SOBOL_SEED_CROSS = 42, 84, 256
XI_WIDE, KAPPA, XI_LOCAL = 0.01, 1.5, 0.005   # EI/PI margins (wide, refinement) and UCB multiplier
W_DISAGREE, W_DIST, W_STD = 0.50, 0.10, 0.05  # conservative-score penalties

# --- Neural-network cross-check ---------------------------------------------
NN_HIDDEN_LAYERS, NN_ACTIVATION, NN_SOLVER = (16, 8), "relu", "lbfgs"
NN_ALPHA, NN_MAX_ITER, NN_MAX_FUN, NN_TOL = 0.5, 2000, 15000, 1e-4
NN_SEEDS = list(range(10))
CROSS_GP_WEIGHT, CROSS_NN_WEIGHT = 0.75, 0.25
PENALTY_GP_DISAGREEMENT, PENALTY_NN_INSTABILITY, PENALTY_DISTANCE = 0.25, 0.05, 0.10

In [2]:
# =============================================================================
# SHARED HELPERS
# =============================================================================


def make_box(center, half_width):
    """Axis-aligned trust region centre +/- half_width, clipped to the valid input range."""
    centre = np.asarray(center, dtype=float)
    return np.column_stack([np.clip(centre - half_width, 0.0, UPPER_BOUND),
                            np.clip(centre + half_width, 0.0, UPPER_BOUND)])


def sobol_candidates(bounds, seed):
    """Scrambled Sobol sample (2**SOBOL_POWER points) scaled into a box."""
    unit = qmc.Sobol(d=len(bounds), scramble=True, seed=seed).random_base2(m=SOBOL_POWER)
    return qmc.scale(unit, bounds[:, 0], bounds[:, 1])


def minmax(series):
    """Min-max normalise a Series to [0, 1] (zeros if constant)."""
    span = series.max() - series.min()
    return (series - series.min()) / span if span > 1e-12 else pd.Series(0.0, index=series.index)


def acquisition(mu, sigma, incumbent, xi):
    """Expected improvement and probability of improvement for maximisation."""
    sigma_safe = np.maximum(sigma, 1e-12)
    improvement = mu - incumbent - xi
    z = improvement / sigma_safe
    return improvement * norm.cdf(z) + sigma_safe * norm.pdf(z), norm.cdf(z)


def build_kernel(base, ls_bounds, **base_kw):
    """Constant * ARD base kernel (one length scale per input)."""
    return C(1.0, (1e-3, 1e3)) * base(length_scale=np.ones(n_features), length_scale_bounds=ls_bounds, **base_kw)


def to_portal(x):
    """Format an input vector for the portal: 6 decimals, hyphen-separated, clipped to [0, 0.999999]."""
    x = np.clip(np.round(np.asarray(x, dtype=float), DECIMALS), 0.0, UPPER_BOUND)
    return "-".join(f"{v:.{DECIMALS}f}" for v in x)


def fmt_table(frame):
    return frame.to_string(index=False, float_format=lambda v: f"{v:.6f}")

In [3]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the available x-columns. The
# incumbent (best observation) and the latest observation are derived from the
# data, so new rounds appended to data.csv flow through every later cell.

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [f"x{i}" for i in range(1, 9) if function_data[f"x{i}"].notna().any()]
function_data = function_data.sort_values("round", kind="stable").reset_index(drop=True)

X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()

n_obs, n_features = X.shape
df = function_data[[*feature_cols, "y", "round"]].rename(columns={"round": "Round"})
df["Stage"] = df["Round"].apply(lambda r: "Initial Data" if r == 0 else f"Week {r}")
df = df[[*feature_cols, "y", "Stage", "Round"]]

# Reference points: incumbent = best y (ties -> earliest round); latest = most recent round.
tied = df.index[df["y"] >= df["y"].max() - TIE_TOL]
inc_idx = df.loc[tied, "Round"].idxmin()
inc_row, latest_row = df.loc[inc_idx], df.iloc[-1]
x_inc, y_inc, inc_round = inc_row[feature_cols].to_numpy(dtype=float), float(inc_row["y"]), int(inc_row["Round"])
x_latest, y_latest, latest_round = latest_row[feature_cols].to_numpy(dtype=float), float(latest_row["y"]), int(latest_row["Round"])
INC, LATEST = f"R{inc_round}", f"R{latest_round}"

# Cross-validation groups: identical inputs share a group and are held out together.
_, groups = np.unique(np.round(X, DECIMALS), axis=0, return_inverse=True)
groups = groups.ravel()
cv = LeaveOneGroupOut()
n_groups = len(np.unique(groups))

print(f"Function {FUNCTION_ID} observations: {n_obs} ({n_groups} distinct inputs)")
print(f"Active features: {feature_cols} | dimensions: {n_features} | rounds: R0-R{df['Round'].max()}")
print(f"y range: {y.min():.6f} to {y.max():.6f}")
print(f"Incumbent: {INC} (y = {y_inc:.6f}) | latest observation: {LATEST} (y = {y_latest:.6f})")
print(f"Tied with incumbent (within {TIE_TOL}): rounds {sorted(df.loc[tied, 'Round'].tolist())}")

Function 8 observations: 51 (49 distinct inputs)
Active features: ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'] | dimensions: 8 | rounds: R0-R11
y range: 5.592193 to 10.000000
Incumbent: R6 (y = 10.000000) | latest observation: R11 (y = 10.000000)
Tied with incumbent (within 1e-09): rounds [6, 10, 11]


In [4]:
# =============================================================================
# TUNED ARD GAUSSIAN PROCESS
# =============================================================================
# The primary surrogate: StandardScaler on X, a Matern 5/2 ARD kernel, a
# Yeo-Johnson target transformation (stabilises skewed y), and grouped
# leave-one-out tuning of the GP noise floor (alpha).

# The pipeline is fed pandas objects on purpose. Pandas and NumPy slice folds
# with different memory layouts, which changes floating-point summation order.
# With 25 optimiser restarts on a flat likelihood (several length scales sit at
# their upper bound) that is enough to flip the selected alpha (1e-4 vs 1e-3).
# Keeping pandas inputs makes the run exactly reproducible.
X_f8_model, y_f8_model = df[feature_cols].copy(), df["y"].copy()

gp_pipeline_f8 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", TransformedTargetRegressor(
        regressor=GaussianProcessRegressor(kernel=build_kernel(Matern, TUNED_LS_BOUNDS, nu=2.5),
                                           n_restarts_optimizer=TUNED_RESTARTS, random_state=GP_SEED),
        transformer=PowerTransformer(method="yeo-johnson")))])

grid_search_f8 = GridSearchCV(gp_pipeline_f8, {"model__regressor__alpha": ALPHA_GRID}, cv=cv,
                              scoring="neg_mean_absolute_error", n_jobs=-1)
grid_search_f8.fit(X_f8_model, y_f8_model, groups=groups)

# best_estimator_ is already refitted on all observations by GridSearchCV
best_pipeline_f8 = grid_search_f8.best_estimator_
best_gp_f8 = best_pipeline_f8.named_steps["model"].regressor_

# Leave-one-out performance of the selected pipeline
y_pred_cv_f8 = cross_val_predict(best_pipeline_f8, X_f8_model, y_f8_model, cv=cv, groups=groups, n_jobs=-1)
mae_f8 = mean_absolute_error(y, y_pred_cv_f8)
rmse_f8 = np.sqrt(mean_squared_error(y, y_pred_cv_f8))
r2_f8 = r2_score(y, y_pred_cv_f8)

print(f"Best alpha: {best_gp_f8.alpha}")
print(f"LOOCV: RMSE={rmse_f8:.6f}, MAE={mae_f8:.6f}, R²={r2_f8:.6f}")
print(f"Fitted kernel: {best_gp_f8.kernel_}")

# ARD length scales: short length scale = the output is sensitive to that input
length_scales_f8 = np.atleast_1d(best_gp_f8.kernel_.k2.length_scale)
importance_f8 = pd.DataFrame({"Dimension": feature_cols, "Length_Scale": length_scales_f8,
                              "Sensitivity_1_over_L": 1.0 / length_scales_f8}
                             ).sort_values("Sensitivity_1_over_L", ascending=False)
print("\nARD SENSITIVITY")
print(fmt_table(importance_f8))

Best alpha: 0.0001
LOOCV: RMSE=0.062039, MAE=0.040192, R²=0.997377
Fitted kernel: 10.6**2 * Matern(length_scale=[21.1, 32.1, 19.1, 40.5, 52.7, 38.7, 24.5, 1e+04], nu=2.5)

ARD SENSITIVITY
Dimension  Length_Scale  Sensitivity_1_over_L
       x3     19.091029              0.052381
       x1     21.068614              0.047464
       x7     24.488322              0.040836
       x2     32.072935              0.031179
       x6     38.725552              0.025823
       x4     40.450092              0.024722
       x5     52.670281              0.018986
       x8  10000.000000              0.000100


In [5]:
# =============================================================================
# LOCAL TRUST-REGION SEARCH AROUND THE INCUMBENT
# =============================================================================
# Compares Matern 5/2 and Matern 3/2 ARD GP acquisitions under the same
# scaling, target transformation and tuned alpha. Acquisitions are computed in
# the GP's transformed target space; posterior means are back-transformed for
# reporting.

# Shared model components from the tuned pipeline
scaler_f8 = best_pipeline_f8.named_steps["scaler"]
target_transformer_f8 = best_pipeline_f8.named_steps["model"].transformer_
gp_m52_f8 = best_gp_f8
best_alpha_f8 = best_gp_f8.alpha

y_f8_transformed = target_transformer_f8.transform(y.reshape(-1, 1)).ravel()
y_inc_transformed_f8 = target_transformer_f8.transform([[y_inc]]).ravel()[0]
X_f8_scaled = scaler_f8.transform(X)

# Matched Matern 3/2 model (same scaling, target transform and alpha)
gp_m32_f8 = GaussianProcessRegressor(kernel=build_kernel(Matern, TUNED_LS_BOUNDS, nu=1.5), alpha=best_alpha_f8,
                                     n_restarts_optimizer=TUNED_RESTARTS, random_state=GP_SEED
                                     ).fit(X_f8_scaled, y_f8_transformed)


def gp_posterior(X_raw):
    """Posterior of both GPs at raw inputs: transformed-space mean/std plus back-transformed mean."""
    Xs = scaler_f8.transform(X_raw)
    out = {}
    for tag, gp in (("M52", gp_m52_f8), ("M32", gp_m32_f8)):
        mean_t, std_t = gp.predict(Xs, return_std=True)
        out[tag] = {"mean_t": mean_t, "std_t": std_t,
                    "mean": target_transformer_f8.inverse_transform(mean_t.reshape(-1, 1)).ravel()}
    return out


# Local trust region: a box around the incumbent, with a Sobol sample of it
# (the incumbent is appended as a reference candidate)
bounds_f8 = make_box(x_inc, WIDE_HALF_WIDTH)
X_candidates_f8 = np.vstack([sobol_candidates(bounds_f8, SOBOL_SEED_WIDE), x_inc])
post = gp_posterior(X_candidates_f8)

print(f"Incumbent {INC}: y = {y_inc:.6f}")
print("Local trust region: " + ", ".join(f"{f} [{lo:.3f}, {hi:.3f}]" for f, (lo, hi) in zip(feature_cols, bounds_f8)))
print(f"Matern 5/2: {gp_m52_f8.kernel_}\nMatern 3/2: {gp_m32_f8.kernel_}\nShared alpha: {best_alpha_f8}")

# Acquisition functions in transformed target space
local_results_f8 = pd.DataFrame(X_candidates_f8, columns=feature_cols)
for tag in ("M52", "M32"):
    ei, pi = acquisition(post[tag]["mean_t"], post[tag]["std_t"], y_inc_transformed_f8, XI_WIDE)
    local_results_f8[f"{tag}_Mean"] = post[tag]["mean"]
    local_results_f8[f"{tag}_Std_T"] = post[tag]["std_t"]
    local_results_f8[f"{tag}_EI"] = ei
    local_results_f8[f"{tag}_PI"] = pi
    local_results_f8[f"{tag}_UCB"] = post[tag]["mean_t"] + KAPPA * post[tag]["std_t"]
local_results_f8["Distance_to_Inc"] = np.linalg.norm(X_candidates_f8 - x_inc, axis=1)

# Acquisition winners under each kernel
criteria_f8 = {f"{tag} {label}": f"{tag}_{col}" for tag in ("M52", "M32")
               for label, col in (("Posterior Mean", "Mean"), ("EI", "EI"), ("PI", "PI"), ("UCB", "UCB"))}
winners_f8 = pd.DataFrame([local_results_f8.loc[local_results_f8[col].idxmax()].rename(method).copy()
                           for method, col in criteria_f8.items()])
winners_f8["Method"] = winners_f8.index
winners_f8 = winners_f8.reset_index(drop=True)
print("\nLOCAL ACQUISITION WINNERS")
print(fmt_table(winners_f8[["Method", *feature_cols, "M52_Mean", "M52_Std_T", "M32_Mean", "M32_Std_T", "Distance_to_Inc"]]))

m52_best = local_results_f8.loc[local_results_f8["M52_Mean"].idxmax(), feature_cols].to_numpy(dtype=float)
m32_best = local_results_f8.loc[local_results_f8["M32_Mean"].idxmax(), feature_cols].to_numpy(dtype=float)
print(f"\nDistance between the kernels' posterior-mean optima: {np.linalg.norm(m52_best - m32_best):.6f}")

# Cross-kernel consensus score: average of min-max normalised posterior means
local_results_f8["Consensus_Mean_Score"] = 0.5 * minmax(local_results_f8["M52_Mean"]) + 0.5 * minmax(local_results_f8["M32_Mean"])
top_consensus_f8 = local_results_f8.nlargest(10, "Consensus_Mean_Score")
print("\nTOP 10 CROSS-KERNEL CONSENSUS CANDIDATES")
print(fmt_table(top_consensus_f8[[*feature_cols, "M52_Mean", "M32_Mean", "Consensus_Mean_Score", "Distance_to_Inc"]]))

Incumbent R6: y = 10.000000
Local trust region: x1 [0.000, 0.200], x2 [0.050, 0.250], x3 [0.030, 0.230], x4 [0.050, 0.250], x5 [0.700, 0.900], x6 [0.400, 0.600], x7 [0.100, 0.300], x8 [0.500, 0.700]
Matern 5/2: 10.6**2 * Matern(length_scale=[21.1, 32.1, 19.1, 40.5, 52.7, 38.7, 24.5, 1e+04], nu=2.5)
Matern 3/2: 6.09**2 * Matern(length_scale=[22.3, 39.8, 20.6, 41.1, 136, 1e+04, 29.8, 1e+04], nu=1.5)
Shared alpha: 0.0001

LOCAL ACQUISITION WINNERS
            Method       x1       x2       x3       x4       x5       x6       x7       x8  M52_Mean  M52_Std_T  M32_Mean  M32_Std_T  Distance_to_Inc
M52 Posterior Mean 0.103388 0.134966 0.120910 0.165386 0.879862 0.505847 0.218925 0.573318 10.001609   0.010788 10.021816   0.014056         0.089661
            M52 EI 0.129796 0.112340 0.107476 0.182418 0.897560 0.503513 0.227334 0.636446  9.998331   0.015834 10.027287   0.024380         0.124379
            M52 PI 0.129796 0.112340 0.107476 0.182418 0.897560 0.503513 0.227334 0.636446  9.998331 

In [6]:
# =============================================================================
# CONSERVATIVE REFINEMENT AROUND THE CONSENSUS REGION
# =============================================================================
# Refines the promising region identified by both Matern kernels while
# controlling distance from the incumbent. The consensus centre is the top
# cross-kernel consensus candidate from the local search above (derived, not
# typed in). Posterior means are back-transformed for interpretation.

consensus_center = top_consensus_f8.iloc[0][feature_cols].to_numpy(dtype=float)
bounds_refine_f8 = make_box(consensus_center, REFINE_HALF_WIDTH)
print("Consensus centre: " + ", ".join(f"{f}={v:.6f}" for f, v in zip(feature_cols, consensus_center)))
print("Refinement region: " + ", ".join(f"{f} [{lo:.3f}, {hi:.3f}]" for f, (lo, hi) in zip(feature_cols, bounds_refine_f8)))

X_refine_f8 = np.vstack([sobol_candidates(bounds_refine_f8, SOBOL_SEED_REFINE), consensus_center, x_inc])
post = gp_posterior(X_refine_f8)

refine_results_f8 = pd.DataFrame(X_refine_f8, columns=feature_cols)
for tag in ("M52", "M32"):
    ei, pi = acquisition(post[tag]["mean_t"], post[tag]["std_t"], y_inc_transformed_f8, XI_LOCAL)
    refine_results_f8[f"{tag}_Mean"] = post[tag]["mean"]
    refine_results_f8[f"{tag}_Std_T"] = post[tag]["std_t"]
    refine_results_f8[f"{tag}_EI"] = ei
    refine_results_f8[f"{tag}_PI"] = pi
refine_results_f8["Distance_to_Inc"] = np.linalg.norm(X_refine_f8 - x_inc, axis=1)

# Cross-model consensus and disagreement (raw-y scale); uncertainty is only a diagnostic (transformed scale)
refine_results_f8["Consensus_Mean"] = 0.5 * (refine_results_f8["M52_Mean"] + refine_results_f8["M32_Mean"])
refine_results_f8["Consensus_Std_T"] = 0.5 * (refine_results_f8["M52_Std_T"] + refine_results_f8["M32_Std_T"])
refine_results_f8["Model_Disagreement"] = (refine_results_f8["M52_Mean"] - refine_results_f8["M32_Mean"]).abs()

# Conservative score: reward the consensus mean, penalise disagreement, distance and (normalised) uncertainty
refine_results_f8["Conservative_Score"] = (refine_results_f8["Consensus_Mean"]
                                           - W_DISAGREE * refine_results_f8["Model_Disagreement"]
                                           - W_DIST * refine_results_f8["Distance_to_Inc"]
                                           - W_STD * minmax(refine_results_f8["Consensus_Std_T"]))

candidate_indices = {"Consensus Mean": refine_results_f8["Consensus_Mean"].idxmax(),
                     "Conservative": refine_results_f8["Conservative_Score"].idxmax(),
                     "M52 PI": refine_results_f8["M52_PI"].idxmax(), "M32 PI": refine_results_f8["M32_PI"].idxmax(),
                     "M52 EI": refine_results_f8["M52_EI"].idxmax(), "M32 EI": refine_results_f8["M32_EI"].idxmax()}
candidate_summary_f8 = pd.DataFrame([refine_results_f8.loc[i].rename(n) for n, i in candidate_indices.items()])
candidate_summary_f8["Candidate"] = candidate_summary_f8.index
candidate_summary_f8 = candidate_summary_f8.reset_index(drop=True)

print("\nREFINEMENT CANDIDATES")
print(fmt_table(candidate_summary_f8[["Candidate", *feature_cols, "M52_Mean", "M32_Mean", "Consensus_Mean",
                                      "Model_Disagreement", "Distance_to_Inc", "Conservative_Score"]]))

Consensus centre: x1=0.114146, x2=0.165905, x3=0.139498, x4=0.199466, x5=0.891536, x6=0.507470, x7=0.195648, x8=0.603178
Refinement region: x1 [0.064, 0.164], x2 [0.116, 0.216], x3 [0.089, 0.189], x4 [0.149, 0.249], x5 [0.842, 0.942], x6 [0.457, 0.557], x7 [0.146, 0.246], x8 [0.553, 0.653]



REFINEMENT CANDIDATES
     Candidate       x1       x2       x3       x4       x5       x6       x7       x8  M52_Mean  M32_Mean  Consensus_Mean  Model_Disagreement  Distance_to_Inc  Conservative_Score
Consensus Mean 0.139327 0.152863 0.129534 0.200267 0.940785 0.495158 0.212521 0.572451 10.000412 10.048284       10.024348            0.047872         0.157612            9.954179
  Conservative 0.100000 0.150000 0.130000 0.150000 0.800000 0.500000 0.200000 0.600000  9.998712  9.998766        9.998739            0.000053         0.000000            9.998712
        M52 PI 0.119856 0.158900 0.130303 0.170470 0.921751 0.497468 0.219308 0.621190 10.002722 10.039266       10.020994            0.036543         0.128624            9.968388
        M32 PI 0.111322 0.148466 0.133223 0.180834 0.938497 0.500431 0.189064 0.651569 10.001136 10.042411       10.021773            0.041276         0.151830            9.965432
        M52 EI 0.124296 0.155078 0.121669 0.189196 0.939508 0.502456 0.229446

## Neural network check

The cells below test a small MLP surrogate against the Gaussian process models above. Attribution note: the neural-network cross-validation diagnostic was drafted with ChatGPT's assistance.

In [7]:
# =============================================================================
# MLP VALIDATION (LEAKAGE-FREE, MULTI-SEED, GROUPED LOOCV)
# =============================================================================
# Validates one MLP specification (configured at the top of the notebook) using
# leave-one-out cross-validation across several random seeds. The scaler is
# fitted inside every fold, so held-out observations cannot influence the
# training-data scaling. The same specification is reused in the final GP+NN
# search through build_f8_mlp().


def build_f8_mlp(seed):
    """Single source of truth for the MLP specification."""
    return Pipeline([("scaler", StandardScaler()),
                     ("mlp", MLPRegressor(hidden_layer_sizes=NN_HIDDEN_LAYERS, activation=NN_ACTIVATION,
                                          solver=NN_SOLVER, alpha=NN_ALPHA, max_iter=NN_MAX_ITER,
                                          max_fun=NN_MAX_FUN, tol=NN_TOL, random_state=seed))])


splits = list(cv.split(X, y, groups))
seed_scores = []
seed_predictions = []
for seed in NN_SEEDS:
    pred = np.zeros(n_obs)
    for train_idx, test_idx in splits:
        pred[test_idx] = build_f8_mlp(seed).fit(X[train_idx], y[train_idx]).predict(X[test_idx])
    seed_predictions.append(pred)
    seed_scores.append((np.sqrt(mean_squared_error(y, pred)), mean_absolute_error(y, pred), r2_score(y, pred)))

seed_scores = np.array(seed_scores)
ensemble_loo_pred = np.vstack(seed_predictions).mean(axis=0)

print(f"MLP {NN_HIDDEN_LAYERS}, {NN_ACTIVATION}, {NN_SOLVER}, alpha={NN_ALPHA}, {len(NN_SEEDS)} seeds")
print(f"Per-seed LOOCV (mean ± std): RMSE={seed_scores[:, 0].mean():.6f} ± {seed_scores[:, 0].std(ddof=1):.6f} | "
      f"MAE={seed_scores[:, 1].mean():.6f} | R²={seed_scores[:, 2].mean():.6f} "
      f"(range {seed_scores[:, 2].min():.4f} to {seed_scores[:, 2].max():.4f})")
print(f"Ensemble LOOCV:  RMSE={np.sqrt(mean_squared_error(y, ensemble_loo_pred)):.6f} | "
      f"MAE={mean_absolute_error(y, ensemble_loo_pred):.6f} | R²={r2_score(y, ensemble_loo_pred):.6f}")
print(f"Tuned GP LOOCV:  RMSE={rmse_f8:.6f} | MAE={mae_f8:.6f} | R²={r2_f8:.6f}")

MLP (16, 8), relu, lbfgs, alpha=0.5, 10 seeds
Per-seed LOOCV (mean ± std): RMSE=0.250590 ± 0.018098 | MAE=0.198195 | R²=0.957006 (range 0.9477 to 0.9649)
Ensemble LOOCV:  RMSE=0.237512 | MAE=0.189929 | R²=0.961557
Tuned GP LOOCV:  RMSE=0.062039 | MAE=0.040192 | R²=0.997377


In [8]:
# =============================================================================
# FINAL GP + NN CROSS-MODEL CONSENSUS SEARCH
# =============================================================================
# Fits an ensemble of the validated MLP specification to all observations and
# uses it as a secondary cross-check against the two GP models. GP models
# remain primary; the NN contributes evidence only after its predictive
# performance has been established by the preceding LOOCV cell.

# Search region: the GP refinement region; reference candidates: incumbent and the GP conservative pick
bounds_consensus_f8 = bounds_refine_f8.copy()
gp_conservative_f8 = candidate_summary_f8.loc[candidate_summary_f8["Candidate"] == "Conservative",
                                              feature_cols].iloc[0].to_numpy(dtype=float)

X_cross_f8 = np.vstack([sobol_candidates(bounds_consensus_f8, SOBOL_SEED_CROSS), x_inc, gp_conservative_f8])
post = gp_posterior(X_cross_f8)

# NN ensemble trained on all observations (one model per seed)
nn_full_predictions = np.vstack([build_f8_mlp(s).fit(X, y).predict(X_cross_f8) for s in NN_SEEDS])

cross_results_f8 = pd.DataFrame(X_cross_f8, columns=feature_cols)
cross_results_f8["M52_Mean"], cross_results_f8["M52_Std_T"] = post["M52"]["mean"], post["M52"]["std_t"]
cross_results_f8["M32_Mean"], cross_results_f8["M32_Std_T"] = post["M32"]["mean"], post["M32"]["std_t"]
cross_results_f8["NN_Mean"], cross_results_f8["NN_Std"] = nn_full_predictions.mean(axis=0), nn_full_predictions.std(axis=0)
cross_results_f8["Distance_to_Inc"] = np.linalg.norm(X_cross_f8 - x_inc, axis=1)
cross_results_f8["GP_Consensus_Mean"] = 0.5 * (cross_results_f8["M52_Mean"] + cross_results_f8["M32_Mean"])
cross_results_f8["GP_Disagreement"] = (cross_results_f8["M52_Mean"] - cross_results_f8["M32_Mean"]).abs()

# GP-primary cross-model score on min-max normalised components
cross_results_f8["Cross_Model_Score"] = (
    CROSS_GP_WEIGHT * minmax(cross_results_f8["GP_Consensus_Mean"])
    + CROSS_NN_WEIGHT * minmax(cross_results_f8["NN_Mean"])
    - PENALTY_GP_DISAGREEMENT * minmax(cross_results_f8["GP_Disagreement"])
    - PENALTY_NN_INSTABILITY * minmax(cross_results_f8["NN_Std"])
    - PENALTY_DISTANCE * minmax(cross_results_f8["Distance_to_Inc"]))

candidate_indices = {"Best GP Consensus": cross_results_f8["GP_Consensus_Mean"].idxmax(),
                     "Best GP+NN Consensus": cross_results_f8["Cross_Model_Score"].idxmax(),
                     "Best NN Mean": cross_results_f8["NN_Mean"].idxmax()}
cross_winners_f8 = pd.DataFrame([cross_results_f8.loc[i].rename(n) for n, i in candidate_indices.items()])
cross_winners_f8["Candidate"] = cross_winners_f8.index
cross_winners_f8 = cross_winners_f8.reset_index(drop=True)

summary_cols = [*feature_cols, "GP_Consensus_Mean", "GP_Disagreement", "NN_Mean", "NN_Std",
                "Distance_to_Inc", "Cross_Model_Score"]
print("GP + NN CROSS-MODEL CONSENSUS")
print(fmt_table(cross_winners_f8[["Candidate", *summary_cols]]))

# Incumbent vs GP conservative candidate (the last two rows appended above)
comparison_points_f8 = cross_results_f8.iloc[-2:].copy()
comparison_points_f8.insert(0, "Point", [f"{INC} incumbent", "GP conservative candidate"])
print(f"\n{INC} VS GP CONSERVATIVE CANDIDATE")
print(fmt_table(comparison_points_f8[["Point", *summary_cols]]))

GP + NN CROSS-MODEL CONSENSUS
           Candidate       x1       x2       x3       x4       x5       x6       x7       x8  GP_Consensus_Mean  GP_Disagreement   NN_Mean   NN_Std  Distance_to_Inc  Cross_Model_Score
   Best GP Consensus 0.127733 0.148718 0.139281 0.192177 0.940737 0.502177 0.214399 0.590069          10.024145         0.045250  9.889608 0.034346         0.150842           0.578231
Best GP+NN Consensus 0.118271 0.138457 0.128467 0.168583 0.921286 0.503217 0.196682 0.629135          10.020284         0.036158  9.921216 0.023783         0.128044           0.614854
        Best NN Mean 0.067724 0.116139 0.101881 0.156724 0.941475 0.509871 0.149686 0.652899           9.996021         0.022383 10.008521 0.022152         0.168720           0.389046

R6 VS GP CONSERVATIVE CANDIDATE
                    Point       x1       x2       x3       x4       x5       x6       x7       x8  GP_Consensus_Mean  GP_Disagreement  NN_Mean   NN_Std  Distance_to_Inc  Cross_Model_Score
             

In [9]:
# =============================================================================
# FINAL RECOMMENDATION AND PORTAL FORMAT
# =============================================================================
# Reports the best observed input (the incumbent) and the leading model-based
# candidate in the exact portal format (6 decimals, hyphen-separated).
# Decision rule: a model-based candidate is recommended only if its predicted gain
# over the best observed y exceeds the tuned GP's leave-one-out MAE (the model's
# typical prediction error). Otherwise the incumbent is the recommended submission,
# because a gain inside the model's error is indistinguishable from noise.

best_cross = cross_winners_f8.loc[cross_winners_f8["Candidate"] == "Best GP+NN Consensus"].iloc[0]
x_best_cross = best_cross[feature_cols].to_numpy(dtype=float)

print(f"Observations used: {n_obs} ({n_groups} distinct inputs) | rounds R0-R{df['Round'].max()} | dimensions: {n_features}")
print(f"Best observed y: {y_inc:.6f} at {INC}")
print(f"Incumbent portal input : {to_portal(x_inc)}")
print(f"GP+NN consensus input  : {to_portal(x_best_cross)}")
print(f"  GP consensus mean = {best_cross['GP_Consensus_Mean']:.6f} | NN mean = {best_cross['NN_Mean']:.6f} | "
      f"distance to incumbent = {best_cross['Distance_to_Inc']:.6f}")
predicted_gain = best_cross["GP_Consensus_Mean"] - y_inc
print(f"  Predicted gain over incumbent (GP consensus mean - best observed y): {predicted_gain:+.6f} "
      f"(GP LOOCV MAE = {mae_f8:.6f})")
recommended = x_best_cross if predicted_gain > mae_f8 else x_inc
print(f"\nRecommended submission  : {to_portal(recommended)} "
      f"({'GP+NN consensus candidate' if predicted_gain > mae_f8 else 'incumbent: predicted gain is within model error'})")

Observations used: 51 (49 distinct inputs) | rounds R0-R11 | dimensions: 8
Best observed y: 10.000000 at R6
Incumbent portal input : 0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000
GP+NN consensus input  : 0.118271-0.138457-0.128467-0.168583-0.921286-0.503217-0.196682-0.629135
  GP consensus mean = 10.020284 | NN mean = 9.921216 | distance to incumbent = 0.128044
  Predicted gain over incumbent (GP consensus mean - best observed y): +0.020284 (GP LOOCV MAE = 0.040192)

Recommended submission  : 0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000 (incumbent: predicted gain is within model error)
